# GwenLand glcuda — T4 Ceiling Wave 2 (Direct Model Fetch)

This Kaggle notebook establishes a reproducible, same-session baseline for the hand-written sm_75 INT8 Tensor Core kernels.

It compares two isolated builds from the same commit:

- baseline: clean commit 3bce8dd7b8aa
- candidate: the current vector-store, 48-byte shared-memory pitch, and B-fragment prefetch patch embedded in this notebook

The notebook deliberately separates correctness, diagnostic microbenchmarks, production glbench, and optional Nsight Compute counters. A microbenchmark result is never promoted to a production performance claim.

Kaggle setup:

1. Accelerator: GPU T4 x1 or T4 x2.
2. Internet: On. The notebook downloads a revision-pinned Qwen2.5-0.5B-Instruct Q4_K_M GGUF directly from the official Qwen Hugging Face repository.
3. No Kaggle model dataset is required. Cell 2 immediately prints MODEL FETCH START, resumes partial transfers, verifies the exact byte count and SHA-256, and caches the model under /kaggle/working/models.
4. The default Wave 2 policy forces Q8 staging in both A/B arms so every measured projection reaches the same MMA path.
5. Run all cells in order.
6. Download glcuda_t4_ceiling_wave2_fetch_results.zip from /kaggle/working.


## 1 · Configuration, T4 gate, and isolated source trees

The first code cell is intentionally strict. It rejects a non-T4 GPU, verifies compute capability 7.5, checks out the exact baseline commit twice, and applies the embedded candidate patch only to the candidate tree.


In [ ]:
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import time

REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
PATCH_SHA256 = "660a59e092e5c573770f9d5d30db543a88da5e9866cac91e195fb51c1fd85afd"
PATCH_GZIP_B64 = """H4sIAAAAAAAEAO1de3PaSBL/359iNlWuQABZ85AQ9voqduLdTW2c5Gzf5bZ8PkqIAbQWEpGEH7WX/ezX8xAIhG3psldlHUu5AGt6WtO/7uluzQwzQ380Qp3O2E+RuzcOvPnQ3Utib++axyEPEn2pn0y7ljFL79CgBNGOHw75HTIHLvWwaxisZzvEwQibps3YTqfTKXWvnVarVe5+r1+jDrHbXdSS769f76C9PcRveHyPbt14Bm8Jinkn5u7QD8conXB08uHi3dkJcr3Uv3FTPwpRErgDNIqjKXpPEPzvpwmKbkPUkdxGUSzrjd2Uox9PTk/biFDbkfwTdIcI6aGfj9Eh+t1iJjo9RtFI8EljdzTyPTTjMQrceehNFtzcrM7UTWP/zkBHQYAyhlDbRYMg8q5RMnFjLu+duFP4El3zMGmjJBLXdjrADZh3rjuKekUg3+OoYTMUR7eijZQg0UICN20iNxxKEaF4RMlOqwIf5kg+dBMfIV2+rucGPEFCgjC6RecXRz+evEXvPqDzn47O4NvpyenHs18AbriDwEjfPBxKRl4UJvMpH6LBfU6d++hogeswjgAt566Nfvjhg9RLgsYRent2dNoZRPNwaAhG0kKotBCasxDx+qvTN9F5dIS+JChJwUKm6N05IOsnoK/7aJ6ixmchd2fq/gpKOzxEx/+6gIYF6kLT0JzeooZz54BaKGmi7HUI+lT6AtRAtdwfT1KJoqr2CQBPJ8IswfDc8ZSHAGRjHEfz2bu3UDtwQ77H2ij1x/q/XdbclzpHyDXhWjLl07572UhN1EK6YvMVKLqFJm4weoVt+AbVXzFkGC12hVAjCjmaQyulyh9lA3ouwUbJPxBsPl+GOQZXl9eDR5uCimyGZhsNMbD6pZP6ARdYIWhVK2tTW0CfIPIKGLXVRwsDm/Q2ktAvtG1ZQtvyXWvbTdGns5Mf3r1/3z8+unjzE9zEZs0DdNf/kuzd9bWlShOfzpMUDTg6ev/+45ujC7DT+QwUKdnEwqqcRgh6baIGvwNLVJWEjQtVtlEobBXdxn6a8lBbyHvZ+fcBG3+IPO4HDWFbe2A16BUKU3+4B61vCkdi2UjZBBiJ0PS56P/DfURM2e8KHbMlqxwXe50hNZxVp2aXVKwu2t354147yLjxE38AajXA0qE3j4P+mE+n/enU7X9xGjvSJIyZG0MvNObgUWb92y9JWyq0S4VC4R1jqVFJGvMxMgZgY7vx98z5y0H+8khcHn1PyerlAbDdjYffMwbXO/K69LBDZLiBPw4R2KkxcKBLQIcQmF8dCCKAAirKzpxzpz+DZ0p5stPSJB9O/nEBWGadeU8bVRY8kmiUggfjnZk/44Efwk2vO0EUzQzFISfPQPAww7b+hsODPImU7TbxJAF8rhXPhDi7sxCsM7v+sIzCMB6UUZrczB0Ooe7MT0X42sCOZdzukkuwJsVNsxNNLdjWDlJ8gqHStlQ2aAW30aXU+dWBVDpmPaF1bFHdj8ULWmMkip6Au4APp63+0Yr2blLB0JhLwxgSLIpJ9+Ch6tkHPlh4bmj7OIgGbgBBwR1DbEpj1HCHNy4EqgSiCNm7HjSV+SSTQJsgke0gVhtZGvbpPDCCSDdkWcycfFvyhc5KU6fRTVa315baeqheT9VeCqBk0GpSMkAlEGLk3/FhU6Fr2+0eoNs14WMFXcmaStwoke/4AdZ3ibfKWGtWRzeZfmnSs5Ojt2jgJjzZX/QQkehIHz995eRCj7EOLaWiFZgIaNHqS6hKR8xXSGQyBeCXlVkBoWVV5hQAkPV0bXu9pg5mqqXrlZRCKC2gBoEW0hcZ8hVsU5F2+EMu3HBTNf4PYEYd1lTi5DBkGQykYEaqTOuabbyR0PTGW4moq+ypR9sQ5HGv2ya2NKiCGUNPM9cVuLjN9QA1ftYpoHyGEIwz1/op9iH7ffO3szNI3R/wsdeDQxPd+K50ttIXgzv0k5THiYEuIFvOmPG7GQQ/X+RiqUjorzmHNFK66BREvO8k8rJwzQhkhPgMNwHPiKD72vuIu+AIM14+sFe+DdzuiIOPhNZcDyAtuZ2IFAZygATa5kUxXNcZLZRrly+wWXX45sGmErxWko8A5sjUrw0EeBNBwtOZMU6VVmSckKbBlvd4/Z26jgaQ35yeHvV/fv/x46escHcGURict/KRis1CgEvhyPHVk7R4QdvC9oPkEKp2J1hT0lUy4ehBTmMkiLJwOMGPsiKaFXucFZasyLouYnjCXIhaLOwW4vVCFSPS1m0slNBcEN9Zgr2vO7DUFde6Uo7eVOrS3Rgk6C309Pns3cWJip4E4zam8FRMrTa2oENKAvkMBinw2f7CU4vsTD79SXftRdPZHKy/kVz7sxk47ttJBHYMz2CQsnaiUSd2Qwgn8kG1iVRml5kMILkwmPNfPryBFioHCXf5rB5/Fq6/pZOBfehnrkgwPn54c9KGLjtP8o98U/UkMOBBdKvjwpo1SaVkVpd3KMfyyQOZqHGNTMMADCDcSP/SfIBRd8Uk1xlhwQjbhkFxkUHBUCXBil1JG5jgggOEO4D+L0PPFM9MV5tZ5w13A2uq7PVB1lixXrpT5aseSlkHHBwq1/4qGzTx5nEsNCchNNaDFetmtonzbiZYdzOKTlhv1v+ko3ncnUDKdVWiQs6niMz9wQqrymoVeD/pWTYyXKjoaYYL/7LMl87mYSiQVulSB2JuBOYvQi7EL9EvAE4Rg3SHMGUYQteZNlDmVHTSY21IGhY2cSQVLkO5chWMqPEzRybaBUex6ip0C/B651/v++upkWoSvEMUPSjkOotSSF6KCYrd1u80y14AfQVV1nmZhJ9aVw8RWJpAxRs1bKjkdqrITdbkJjWU25FyU7OK3HRNblo/uamp5CZV5GZrcrMayk2U3KyK3Naa3FYN5VZ+jVbya/aa3HYN5VZ+jVbya901ubs1lNuRj6DM7LWxuUnwhTTL7PdEpJliHLohx6k4gqe1MEqbaMp5mqAJj3mWDokkSIZzNaISQU0xzMt19B24sZHch17xGe5//dywRFcPpMmxG/lByfrjvJRAP2gXa9J29kE28GXt7IMoxBl4VAcQp2RpafIZRAM8dYdiJGZlSEk9vsi3bCRMDNjcwqN2NmiXDTu1ESu2QhUOsRq3EwakhovSLB0T+FzKoisgGWHzEYoWUzSLpHVBc0NWGf0mOCnarwfKlD7j/YJlm5mMmWyrT3DYXGJANYagJ0uYLTPbrLdAET8nCB+jyCCkpSAkivZr0SdUQ075KXF54afefvxwcvCMQGMlQLNKgcYU7TaAZpcArVsKNFvRbgNoTgnQeqVAcxTtFoBGSoQFUiosyAE5oN0G0EoEAlIqEBCiaLcBtBKBgJQKBIQp2q9y5CgTVScKMU91QqFmy1nPWiZlmk4sYVAjrK4XR0miljbIRQBizlcuLHKRhUlHTQR7k3l4LVZPqdUxiV7FkkhO6a1YQuCHctZEDK6qId7Un/LEQG+iMI0hp94XA8e7zuEhwOWHu5TIb2IFxfeHcN+2ZLW6DMMNIDl1U2ilmHTJLbkw0I8nH07O5KKMBp/6aV9MyRiz+2Z+RRQlnVEUDNE8jKMgQH6CZnF04w4CMZ42ngeQp6OzbGoI/d4jbTXH1SNYrIXo/AX9rpZE/beMMKWYLDgJ+WS2neydn6LG75a5iyLPm8/c0LtvHsjHigEPvQm6FOshOrOJm/DBFTraO0ZD7oFVAuwTDmSxgl0sTwp52hlwN1XzVo56hFNL5MTyNXHx+LNaBLVcHPXHL+qQ5taT0/QWwQVzE6/HlzXkSDYta8gVryxreGrphoN7y2UNclHNY2s3HmGECXHUIpBNjKovkMAmYSsN+6YVElavK+bwbdxr9zZAnxsPlqYJDTj4c4p+e6fou7KLtrqMLntqfrJxPQoSKR2TczorwMqlfU8tqMlic26eRNOpoR0gzCazlhgwdcsNdiRX391KE0INbbHigpKyucGslrxWzWrBCbrvJk5r0Gs26t1aY9SSfgR6yQiSlM01lXWwDA4I8Nk4FsBxKUquFGDsQK9160otWb310bsj809FPSdFWWpVYtcqKAr/qajnpChbZijdHi4oivypqGekKEct03esb5qINms4Fe1YSvLut0xFQ3JZQ8m7SvLet0xGY7OG09FOT0rew98yHY3NGk5I97CSnH7LhDQ2azgl3VMerlfJw61PSWOzhpPSPeXhepU83PqkNDZrOC3dUx6uV8nDOQXJnRpKrjwcROQqovcKovfqJzrIrGWv5OSwWVhTa9ZReKqFr5bJFVK54oriOghvaeEruTpcyOZwDbM5EFoLX8nb4UJCh2uY0IHQSnhcyd/hQk6Ha5jTgdBa+GoOr5DW4RqmdSC0Fr6awytkdriGmR0IrYWv5vAKyR2uYXIHjdbCV3N4hfwO1zC/A6GV8KSawytkeLiOGR7RDo9UcnikkOGROmZ4RDs8UsnhkUKGR+qY4RHt8Ei14bpChkfqmOER7fBIJYdHChkeqWOGR7TDo5UcHilkeKSOGR7VDo9Wc3iFDI/UMcOj2uHRag6vkOGROmZ4VDs8Ws3hFTI8UscMj2qHR6s5vEKGR+qY4VHt8Fg1h1fI8EgdMzymHR6r5PBoIcOjdczwmHZ4rJLDo4UMj9Yxw2MObovleS1smbRNIFxv1a+b1kD+5hXZn5/Zkux6/KDp2aFWi180PTvUavGTpmeHWi1+0/TcUKvHj5qeHWq1+FXTs0Ptf/Kzpv971EpEA1IqGshNBYB2K1ArEQ1IqWggt8wE2m1AjZaIBrRUNJBNB9qtQK1ENKClooG8M9BuBWologEtFQ3kT52AditQKxENaKloIAdCgHYrUCsRDWipaEAdRbsNqLES0YCVigbMVLRbgVqJaMBKRQNGFO1WoFYiGrBS0YAxRbsVqJWIBqxUNGC2ot0K1EpEA1YqGjBH0W4DalaJaGCVigaWqWi3ArUS0cAqFQ0somi3ArUS0cAqFQ0spmi3ArUS0cAqFQ0sW9FuBWolooFVKhpYjqLdBtTsEtHALhUNbFPRbgVqJaKBXSoa2ETRbgVqJaKBXSoa2EzRbgVqJaKBXSoa2Lai3QrUSkQDu1Q0sB1Fuw2odUtEg26paNA1Fe1WoFYiGnRLRYMuUbSLfab1nm6bNnwcPnGG7zQaGnGy+ShdVaZP7CUj6hGbGsYAM7vr0CdP7NW1HzyoV5frH0P35OFg4pOJIxwRlKKUJ2mCflPSSPSThMdpn3/5rvHp4h/989OuZUxdeahW4+VvL5uGF83DtNFso2Lx12Vx82CdZY6fF4Wp64dJ44WRuvGYp2JjvK71omyt6dSV27yrzf340Jg6oXONbSOOboEsEEZoJI78o0SybWVss/3m0Vt5niuauX4stpV0h7+6nthtT5yn63TkvjqavdyScxClE727Y7LCTZ5VCnYATUNiE/sb7qVRDNYl9qsXR+rqVZmCbxjJ3QXdWN1W0iTGkt1j4L9Yt9cXOV2IjQEpycuZwffdBvw29h8F04poR4hpJI46Yosiub+iAEuCAfxGge+lnVHM+eKITeggCtipO1thJuV303nsBsF9Bu2iWsy/zH2xFZLNOgNf7e8vz+JREC0Pj9PcUj6dpYKA2qqFqm2Jm/rJyOdid05fbNcptgsFJaT3BZQ3WtbyQM5Vq3mqitqjclOdh5W53Icpr0kMmmR5NgF0j+k8RQM3vE7QIfo79/b3b0G4vufOXM9P7xurmheQyi0b+/4QgVGahuFAD1+UZzTiUGNVzNaLxUvez5jNk0mj0Vjwk9tWyW0dxdcm2hNvu2u2J15fl//mviqmSQS4zEPQ8SDgjXxNVT7kw/ms8QCWiiTgoUBLBAipgYXNqf6YRnOwBnV+maggD7h+sY6SsGmFgFhnvAEiYVl9tbuWILwUoRJUBNkbYVebQMu1s7HY5auV5yPQAh7m43it2PpxzvXIkxKRCz1JHMu9dnaY7GPZIYjqCER1bsYKO3nkoug6vhAqjVZOFhPdV3QY4JSIE5tFGb/zE1lh4Maxz+OyDuvRbWfzNr/BcUmWqwhlNyiivrzl02eYvWhuqJ+1ZLWILP/dFJg2eNaX/zRfbgxiG2njlyD7izdn739At9E8GIoTr2P+K8QPdfzeLL1zkxd5dv5IuoPzaMob4vjiJviDBWdxIWk0DR7Op+JETOhZxggSi8a/G/02Cpr/Rt8Fhp/03cTzfYjQYL//AaM+4yV+gAAA"""

# Official Qwen GGUF, pinned so every A/B session reads identical bytes.
NOTEBOOK_BUILD = "wave2-fetch-v2-early"
HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491_400_032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
MODEL_PATH = ""  # populated by the pinned fetch cell

# Decision-grade defaults. Keep these fixed between reruns.
TARGET_PREFILL_TPS = 15_000.0
T4_INT8_TMAC_S = 65.0  # 130 TOPS when multiply and add are counted separately
FORCE_Q8_FOR_PTX = True
MICRO_REPEATS = 3
PRODUCTION_REPEATS = 2
COLD_ITERS = 3
WARMUP_ITERS = 1
MEASURE_ITERS = 5
INCLUDE_R256 = True
RUN_NCU = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave2-{RUN_ID}"
META_REPO = ROOT / "meta"
BASE_DIR = ROOT / "base"
CAND_DIR = ROOT / "candidate"
RESULTS = ROOT / "results"
BASE_TARGET = ROOT / "target-base"
CAND_TARGET = ROOT / "target-candidate"
for path in (ROOT, RESULTS):
    path.mkdir(parents=True, exist_ok=False)

def run(cmd, cwd=None, env=None, timeout=7200, check=True):
    merged = dict(os.environ)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
        stdin=subprocess.DEVNULL,
    )
    if check and proc.returncode:
        tail = (proc.stdout + "\n" + proc.stderr)[-5000:]
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(map(str, cmd))}\n{tail}")
    return proc

def save_log(name, proc):
    path = RESULTS / name
    path.write_text(
        f"$ {' '.join(map(str, proc.args))}\nexit={proc.returncode}\n\n"
        f"--- stdout ---\n{proc.stdout}\n--- stderr ---\n{proc.stderr}",
        encoding="utf-8",
    )
    return path

gpu_proc = run([
    "nvidia-smi",
    "--query-gpu=index,name,compute_cap,memory.total,driver_version",
    "--format=csv,noheader,nounits",
], timeout=60)
gpu_rows = [line.strip() for line in gpu_proc.stdout.splitlines() if line.strip()]
if not gpu_rows:
    raise SystemExit("No NVIDIA GPU is visible. Enable a Kaggle GPU accelerator.")
print("Visible GPUs:")
for row in gpu_rows:
    print(" ", row)

first = [x.strip() for x in gpu_rows[0].split(",")]
if len(first) < 5 or "T4" not in first[1] or first[2] != "7.5":
    raise SystemExit(f"GPU 0 must be NVIDIA T4 compute capability 7.5; got: {gpu_rows[0]}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Pinned CUDA_VISIBLE_DEVICES=0")

if shutil.which("cargo") is None:
    installer = run([
        "bash", "-lc",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
        "sh -s -- -y --profile minimal",
    ], timeout=1200)
    save_log("rustup-install.log", installer)
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    raise SystemExit("Rust installation did not expose cargo.")

clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
save_log("git-clone.log", clone)
have_rev = run(["git", "cat-file", "-e", f"{BASE_REV}^{{commit}}"], cwd=META_REPO, check=False)
if have_rev.returncode:
    fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
    save_log("git-fetch-base.log", fetch)

run(["git", "worktree", "add", "--detach", BASE_DIR, BASE_REV], cwd=META_REPO)
run(["git", "worktree", "add", "--detach", CAND_DIR, BASE_REV], cwd=META_REPO)
actual_base = run(["git", "rev-parse", "HEAD"], cwd=BASE_DIR).stdout.strip()
if actual_base != BASE_REV:
    raise SystemExit(f"Baseline checkout mismatch: expected {BASE_REV}, got {actual_base}")

patch_bytes = gzip.decompress(base64.b64decode(PATCH_GZIP_B64, validate=True))
actual_patch_sha = hashlib.sha256(patch_bytes).hexdigest()
if actual_patch_sha != PATCH_SHA256:
    raise SystemExit(f"Embedded patch digest mismatch: {actual_patch_sha}")
patch_path = RESULTS / "candidate.patch"
patch_path.write_bytes(patch_bytes)
run(["git", "apply", "--check", patch_path], cwd=CAND_DIR)
run(["git", "apply", "--whitespace=nowarn", patch_path], cwd=CAND_DIR)

base_status = run(["git", "status", "--porcelain"], cwd=BASE_DIR).stdout.strip()
cand_status = run(["git", "status", "--porcelain"], cwd=CAND_DIR).stdout.strip()
if base_status:
    raise SystemExit(f"Baseline tree is dirty:\n{base_status}")
if not cand_status:
    raise SystemExit("Candidate tree is unexpectedly identical to baseline.")

ptx_path = CAND_DIR / "glcuda/src/kernels/glcuda_sm75.ptx"
ptx = ptx_path.read_text(encoding="ascii")
markers = {
    "vector_store_pairs": ptx.count("st.global.v2.f32"),
    "scalar_f32_stores": ptx.count("st.global.f32"),
    "u64_shared_stages": ptx.count("st.shared.u64"),
    "mma_instructions": ptx.count("mma.sync.aligned.m8n8k16"),
    "sm_a_3072": ptx.count("sm_a[3072]"),
    "sm_a_12288": ptx.count("sm_a[12288]"),
}
expected = {
    "vector_store_pairs": 40,
    "scalar_f32_stores": 0,
    "u64_shared_stages": 5,
    "mma_instructions": 80,
    "sm_a_3072": 1,
    "sm_a_12288": 1,
}
if markers != expected:
    raise SystemExit(f"Candidate structural markers changed:\nexpected={expected}\nactual={markers}")

print(f"Baseline  {actual_base}")
print(f"Patch     {actual_patch_sha}")
print(f"Run root  {ROOT}")
print("Candidate structural markers:", markers)


## 2 · Fetch the pinned production model

This cell downloads the official Qwen2.5-0.5B-Instruct Q4_K_M GGUF directly. The repository revision, expected size, and LFS SHA-256 are pinned in the configuration cell. A valid cached copy is reused; a partial download is resumed. The model fetch is outside glbench so the benchmark remains observation-only.


In [ ]:
print(f"MODEL FETCH START [{NOTEBOOK_BUILD}]")
sys.stdout.flush()

import urllib.error
import urllib.request

MODEL_CACHE = WORK / "models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
model_dest = MODEL_CACHE / HF_FILENAME
model_part = model_dest.with_name(model_dest.name + ".part")
model_url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def park_invalid(path, reason):
    parked = path.with_name(path.name + f".invalid-{reason}-{RUN_ID}")
    path.replace(parked)
    print(f"Parked invalid cache file: {parked}")

def validate_model(path):
    if not path.is_file():
        return False, "missing"
    size = path.stat().st_size
    if size != HF_EXPECTED_BYTES:
        return False, f"size-{size}"
    with path.open("rb") as handle:
        if handle.read(4) != b"GGUF":
            return False, "magic"
    digest = file_sha256(path)
    if digest != HF_EXPECTED_SHA256:
        return False, f"sha256-{digest[:12]}"
    return True, digest

valid, detail = validate_model(model_dest)
if valid:
    print(f"Reusing verified model: {model_dest}")
else:
    if model_dest.exists():
        park_invalid(model_dest, detail)
    if model_part.exists() and model_part.stat().st_size > HF_EXPECTED_BYTES:
        park_invalid(model_part, f"oversize-{model_part.stat().st_size}")
    if model_part.exists() and model_part.stat().st_size == HF_EXPECTED_BYTES:
        partial_valid, partial_detail = validate_model(model_part)
        if partial_valid:
            model_part.replace(model_dest)
        else:
            park_invalid(model_part, partial_detail)
    if not model_dest.exists():
        start = model_part.stat().st_size if model_part.exists() else 0
        headers = {"User-Agent": "GwenLand-Wave2-Kaggle/1.0"}
        if start:
            headers["Range"] = f"bytes={start}-"
            print(f"Resuming model download at {start / 2**20:.1f} MiB")
        else:
            print(f"Downloading {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}")
        request = urllib.request.Request(model_url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
        except urllib.error.HTTPError as exc:
            raise SystemExit(f"Model download HTTP {exc.code}: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        except urllib.error.URLError as exc:
            raise SystemExit(f"Model download connection failed: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        status = getattr(response, "status", response.getcode())
        if start and status != 206:
            print(f"Server ignored Range (HTTP {status}); restarting the partial download.")
            start = 0
        mode = "ab" if start and status == 206 else "wb"
        downloaded = start
        last_print = time.monotonic()
        with response, model_part.open(mode) as output:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                output.write(block)
                downloaded += len(block)
                now = time.monotonic()
                if now - last_print >= 5:
                    pct = 100.0 * downloaded / HF_EXPECTED_BYTES
                    print(f"  {downloaded / 2**20:.1f} / {HF_EXPECTED_BYTES / 2**20:.1f} MiB ({pct:.1f}%)")
                    last_print = now
        part_valid, part_detail = validate_model(model_part)
        if not part_valid:
            park_invalid(model_part, part_detail)
            raise SystemExit(f"Downloaded GGUF failed integrity validation: {part_detail}")
        model_part.replace(model_dest)
valid, digest = validate_model(model_dest)
if not valid:
    raise SystemExit(f"Final GGUF validation failed: {digest}")
MODEL_PATH = str(model_dest)
MODEL_FETCH = {
    "repo": HF_REPO,
    "revision": HF_REVISION,
    "filename": HF_FILENAME,
    "url": model_url,
    "bytes": model_dest.stat().st_size,
    "sha256": digest,
    "path": MODEL_PATH,
}
(RESULTS / "model-fetch.json").write_text(json.dumps(MODEL_FETCH, indent=2), encoding="utf-8")
print(json.dumps(MODEL_FETCH, indent=2))


## 3 · PTX assembly and resource report

Both source trees are assembled directly with ptxas for sm_75 before Rust builds begin. The raw verbose output is archived; register count, stack usage, spills, and static shared memory must be read from those logs rather than inferred from PTX declarations.


In [ ]:
ptxas = shutil.which("ptxas")
if ptxas is None:
    candidates = sorted(Path("/usr/local").glob("cuda*/bin/ptxas"), reverse=True)
    ptxas = str(candidates[0]) if candidates else None
if ptxas is None:
    raise SystemExit("ptxas is required for Wave 2 but was not found in the Kaggle image.")

tool_versions = {}
for name, cmd in {
    "nvidia_smi": ["nvidia-smi"],
    "rustc": ["rustc", "--version", "--verbose"],
    "cargo": ["cargo", "--version"],
    "ptxas": [ptxas, "--version"],
}.items():
    p = run(cmd, check=False, timeout=120)
    tool_versions[name] = (p.stdout + p.stderr).strip()
    print(f"--- {name} ---\n{tool_versions[name][:1500]}")

def assemble(label, src):
    cubin = RESULTS / f"{label}-glcuda-sm75.cubin"
    ptx_file = src / "glcuda/src/kernels/glcuda_sm75.ptx"
    p = run([ptxas, "-arch=sm_75", "-v", ptx_file, "-o", cubin], cwd=src, timeout=600, check=False)
    save_log(f"ptxas-{label}.log", p)
    text = p.stdout + "\n" + p.stderr
    print(f"\n--- ptxas {label} ---\n{text}")
    if p.returncode:
        raise SystemExit(f"ptxas failed for {label}; see {RESULTS / ('ptxas-' + label + '.log')}")
    if "spill stores" not in text or "spill loads" not in text:
        print("WARNING: this ptxas version did not print explicit spill counters.")
    return {"cubin": str(cubin), "log": text}

PTXAS = {
    "base": assemble("base", BASE_DIR),
    "candidate": assemble("candidate", CAND_DIR),
}
PTXAS_OK = True


## 4 · Correctness gate on the real T4

This gate runs both baseline and candidate GPU suites serially. A green Cargo result containing SKIP: no CUDA driver/device is treated as a failure. Timing cells must not run unless every suite really executed on the device.


In [ ]:
def cargo_env(target):
    return {
        "CARGO_TARGET_DIR": str(target),
        "RUST_BACKTRACE": "1",
        "CUDA_VISIBLE_DEVICES": "0",
    }

def cargo_run(label, src, target, args, log_name, timeout=7200):
    p = run(["cargo", *args], cwd=src, env=cargo_env(target), timeout=timeout, check=False)
    save_log(log_name, p)
    hay = p.stdout + "\n" + p.stderr
    if p.returncode:
        raise SystemExit(f"{label} failed (exit {p.returncode}); see {RESULTS / log_name}\n{hay[-3000:]}")
    return hay

TEST_RESULTS = {}
for label, src, target in [
    ("base", BASE_DIR, BASE_TARGET),
    ("candidate", CAND_DIR, CAND_TARGET),
]:
    print(f"\n=== {label}: host library tests ===")
    host = cargo_run(
        label, src, target,
        ["test", "--locked", "-p", "glcuda", "--release", "--lib", "--", "--nocapture"],
        f"test-{label}-lib.log",
    )
    TEST_RESULTS[f"{label}_lib"] = host

    for suite in ("parity", "forward", "graph_replay"):
        print(f"=== {label}: {suite} ===")
        hay = cargo_run(
            f"{label}/{suite}", src, target,
            ["test", "--locked", "-p", "glcuda", "--release", "--test", suite,
             "--", "--test-threads=1", "--nocapture"],
            f"test-{label}-{suite}.log",
        )
        if "SKIP: no CUDA driver/device" in hay:
            raise SystemExit(f"{label}/{suite} silently skipped CUDA device tests.")
        matches = re.findall(r"test result: (ok|FAILED)\. (\d+) passed; (\d+) failed", hay)
        if not matches or any(state != "ok" or int(failed) != 0 for state, _, failed in matches):
            raise SystemExit(f"Could not prove {label}/{suite} passed on hardware.")
        TEST_RESULTS[f"{label}_{suite}"] = {
            "summaries": matches,
            "skip_count": hay.count("SKIP: no CUDA driver/device"),
        }
        print(" ", matches[-1])

for label, src, target in [
    ("base", BASE_DIR, BASE_TARGET),
    ("candidate", CAND_DIR, CAND_TARGET),
]:
    cargo_run(
        f"{label}/bench build", src, target,
        ["build", "--locked", "--release", "-p", "glcuda", "--example", "bench"],
        f"build-{label}-bench.log",
    )
    cargo_run(
        f"{label}/glbench build", src, target,
        ["build", "--locked", "--release", "-p", "glbench"],
        f"build-{label}-glbench.log",
    )

BINS = {
    "base": {
        "bench": BASE_TARGET / "release/examples/bench",
        "glbench": BASE_TARGET / "release/glbench",
    },
    "candidate": {
        "bench": CAND_TARGET / "release/examples/bench",
        "glbench": CAND_TARGET / "release/glbench",
    },
}
for arm, bins in BINS.items():
    for kind, path in bins.items():
        if not path.exists():
            raise SystemExit(f"Missing {arm} {kind} binary: {path}")

CORRECTNESS_OK = True
print("\nCorrectness gate passed for clean baseline and patched candidate on the real T4.")


## 5 · Interleaved diagnostic GEMM A/B

The repository's glcuda bench is diagnostic evidence only. Each executable contains its own PTX at compile time. Runs are interleaved and the full raw output is archived. The extracted gemm-phaseb rows compare the 64-row and r256 paths on fixed 512-row work.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

MICRO = {"base": [], "candidate": []}
for repeat in range(MICRO_REPEATS):
    order = ("base", "candidate") if repeat % 2 == 0 else ("candidate", "base")
    for arm in order:
        t0 = time.time()
        p = run([BINS[arm]["bench"]], cwd=BASE_DIR if arm == "base" else CAND_DIR,
                env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=7200, check=False)
        save_log(f"micro-{repeat}-{arm}.log", p)
        hay = p.stdout + "\n" + p.stderr
        if p.returncode:
            raise SystemExit(f"Microbench {arm} repeat {repeat} failed; see archived log.")
        phase = [line for line in hay.splitlines() if line.startswith("[gemm-phaseb ") and "512-row" in line]
        reuse = [line for line in hay.splitlines() if line.startswith("[gemm-reuse ") and " in=" in line]
        if len(phase) != 2:
            raise SystemExit(f"Expected two gemm-phaseb rows, found {len(phase)} in {arm} repeat {repeat}.")
        MICRO[arm].append({"repeat": repeat, "phaseb": phase, "reuse": reuse})
        print(f"{arm:9s} repeat {repeat} ({time.time()-t0:.1f}s)")
        for line in phase:
            print(" ", line)

phase_re = re.compile(
    r"\[gemm-phaseb\s+(.+?)\]\s+512-row chunk: "
    r"8x64\s+([0-9.]+)us\s+\|\s+4x128\s+([0-9.]+)us.*?\|\s+"
    r"2x256\s+([0-9.]+)us"
)
MICRO_TABLE = []
for arm, runs in MICRO.items():
    by_label = {}
    for rec in runs:
        for line in rec["phaseb"]:
            m = phase_re.search(line)
            if not m:
                raise SystemExit(f"Could not parse gemm-phaseb row: {line}")
            label = m.group(1).strip()
            by_label.setdefault(label, {"t64": [], "t128": [], "t256": []})
            for key, value in zip(("t64", "t128", "t256"), m.groups()[1:]):
                by_label[label][key].append(float(value))
    for label, values in by_label.items():
        MICRO_TABLE.append({
            "arm": arm,
            "shape": label,
            **{key: statistics.median(v) for key, v in values.items()},
        })

print("\nDiagnostic medians (microseconds per 512-row chunk):")
for row in MICRO_TABLE:
    print(row)


## 6 · Production glbench A/B

Production measurement is the decision authority. The preceding cell provides one revision-pinned, SHA-256-verified GGUF through MODEL_PATH. With FORCE_Q8_FOR_PTX enabled, both baseline and candidate stage the same Q4/K-quant weights as Q8_0 so the comparison measures these MMA kernels instead of silently falling back to per-token GEMV.

Four arms are available:

- clean baseline, forced-Q8, default 64-row path
- patched candidate, forced-Q8, default 64-row path
- clean baseline, forced-Q8 with GLCUDA_R256=1
- patched candidate, forced-Q8 with GLCUDA_R256=1

The r256 arms can be disabled in the configuration cell. Run order rotates between repeats to reduce monotonic thermal drift.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

if MODEL_PATH:
    model = Path(MODEL_PATH)
    candidates = [model]
else:
    candidates = sorted(Path("/kaggle/input").rglob("*.gguf")) if Path("/kaggle/input").is_dir() else []

if len(candidates) != 1:
    print(f"Production gate pending: expected exactly one GGUF, found {len(candidates)}.")
    for path in candidates[:20]:
        print(" ", path)
    print("Set MODEL_PATH in cell 1 and rerun this cell through the artifact cell.")
    PROD_OK = False
    PROD_RECORDS = []
    PROD_SUMMARY = []
else:
    model = candidates[0]
    if not model.is_file() or model.stat().st_size < 10_000_000:
        raise SystemExit(f"Model path is not a plausible GGUF: {model}")
    if FORCE_Q8_FOR_PTX:
        print("Production policy: GLCUDA_FORCE_Q8=1 for both A/B builds.")

    prompt_unit = (
        "Measure this deterministic systems prompt carefully. Explain how token-parallel "
        "integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
    )
    FIXED_PROMPT = prompt_unit * 8
    force_env = {"GLCUDA_FORCE_Q8": "1"} if FORCE_Q8_FOR_PTX else {}
    arms = [
        ("base", "base", dict(force_env)),
        ("candidate", "candidate", dict(force_env)),
    ]
    if INCLUDE_R256:
        arms += [
            ("base_r256", "base", {**force_env, "GLCUDA_R256": "1"}),
            ("candidate_r256", "candidate", {**force_env, "GLCUDA_R256": "1"}),
        ]
    arm_map = {label: (build, env) for label, build, env in arms}

    def percentile(values, q):
        values = sorted(values)
        if not values:
            return None
        index = (len(values) - 1) * q
        lo, hi = math.floor(index), math.ceil(index)
        if lo == hi:
            return values[lo]
        return values[lo] * (hi - index) + values[hi] * (index - lo)

    def session_prefill(path, expected_iters=MEASURE_ITERS):
        data = json.loads(path.read_text(encoding="utf-8"))
        engine_blob = json.dumps(data.get("engine", {}), sort_keys=True).lower()
        if "glcuda" not in engine_blob:
            raise RuntimeError(f"Session did not record glcuda engine: {data.get('engine')}")
        iterations = data.get("measurements", {}).get("iterations", [])
        samples = []
        prompt_counts = []
        for item in iterations:
            ms = float(item.get("prefill_ms", 0.0))
            ntok = int(item.get("prompt_tokens", 0))
            if ms > 0 and ntok > 0:
                samples.append(ntok * 1000.0 / ms)
                prompt_counts.append(ntok)
        if len(samples) != expected_iters:
            raise RuntimeError(f"Expected {expected_iters} prefill samples, found {len(samples)}")
        if len(set(prompt_counts)) != 1:
            raise RuntimeError(f"Prompt token count changed within a session: {prompt_counts}")
        return {
            "samples": samples,
            "prompt_tokens": prompt_counts[0],
            "p50": percentile(samples, 0.50),
            "p90": percentile(samples, 0.90),
            "p99": percentile(samples, 0.99),
        }

    PROD_RECORDS = []
    for repeat in range(PRODUCTION_REPEATS):
        rotated = arms[repeat % len(arms):] + arms[:repeat % len(arms)]
        for label, build_arm, extra_env in rotated:
            archive = RESULTS / f"glbench-{repeat}-{label}.json"
            cmd = [
                BINS[build_arm]["glbench"], "run",
                "--engine", "glcuda",
                "--model", model,
                "--prompt", FIXED_PROMPT,
                "--tokens", "1",
                "--cold-iters", str(COLD_ITERS),
                "--warmup", str(WARMUP_ITERS),
                "--iters", str(MEASURE_ITERS),
                "--temperature", "0",
                "--seed", "42",
                "--kind", "prefill",
                "--out", archive,
            ]
            p = run(cmd, cwd=BASE_DIR if build_arm == "base" else CAND_DIR,
                    env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
            save_log(f"glbench-{repeat}-{label}.log", p)
            if p.returncode:
                raise SystemExit(f"glbench {label} repeat {repeat} failed; see archived log.")
            hay = p.stdout + "\n" + p.stderr
            if FORCE_Q8_FOR_PTX and "GLCUDA_FORCE_Q8:" not in hay:
                raise SystemExit(f"glbench {label} did not confirm the forced-Q8 arm.")
            stats = session_prefill(archive)
            rec = {"repeat": repeat, "arm": label, "archive": str(archive), **stats}
            PROD_RECORDS.append(rec)
            print(f"{label:15s} repeat {repeat}: "
                  f"P50 {stats['p50']:.1f}, P90 {stats['p90']:.1f}, P99 {stats['p99']:.1f} tok/s")

    PROD_SUMMARY = []
    for label, _, _ in arms:
        rows = [x for x in PROD_RECORDS if x["arm"] == label]
        PROD_SUMMARY.append({
            "arm": label,
            "session_p50_median": statistics.median(x["p50"] for x in rows),
            "session_p50_min": min(x["p50"] for x in rows),
            "session_p50_max": max(x["p50"] for x in rows),
            "sessions": len(rows),
        })

    print("\nProduction summary:")
    for row in PROD_SUMMARY:
        print(row)

    # One separate diagnostic run captures event-based stage telemetry. It is
    # never folded into the decision-grade A/B distribution above.
    candidate_labels = [x for x in ("candidate", "candidate_r256") if x in arm_map]
    best_label = max(
        candidate_labels,
        key=lambda name: next(r["session_p50_median"] for r in PROD_SUMMARY if r["arm"] == name),
    )
    best_build, best_env = arm_map[best_label]
    telemetry_archive = RESULTS / f"telemetry-{best_label}.json"
    telemetry_cmd = [
        BINS[best_build]["glbench"], "run",
        "--engine", "glcuda", "--model", model, "--prompt", FIXED_PROMPT,
        "--tokens", "1", "--cold-iters", "0", "--warmup", "1",
        "--iters", "1", "--temperature", "0", "--seed", "42",
        "--kind", "prefill", "--out", telemetry_archive,
    ]
    tp = run(telemetry_cmd, cwd=CAND_DIR,
             env={"CUDA_VISIBLE_DEVICES": "0", **best_env, "GLCUDA_TELEMETRY": "1"},
             timeout=14400, check=False)
    save_log(f"telemetry-{best_label}.log", tp)
    if tp.returncode:
        raise SystemExit(f"Telemetry run failed for {best_label}.")
    telemetry_json = json.loads(telemetry_archive.read_text(encoding="utf-8"))
    prefill_telemetry = (telemetry_json.get("telemetry") or {}).get("prefill") or {}
    stages = prefill_telemetry.get("stages") or []
    if not stages:
        raise SystemExit("GLCUDA_TELEMETRY produced no prefill stages.")
    profile_stats = session_prefill(telemetry_archive, expected_iters=1)
    gemm_names = {"qkv", "attn_out", "ffn_gate_up", "ffn_down"}
    stage_total_ms = sum(float(s.get("total_ms", 0.0)) for s in stages)
    gemm_ms = sum(float(s.get("total_ms", 0.0)) for s in stages if s.get("name") in gemm_names)
    gemm_share = gemm_ms / stage_total_ms if stage_total_ms > 0 else 0.0
    measured_best_tps = next(
        r["session_p50_median"] for r in PROD_SUMMARY if r["arm"] == best_label
    )
    prompt_tokens = profile_stats["prompt_tokens"]
    target_prefill_ms = prompt_tokens * 1000.0 / TARGET_PREFILL_TPS
    measured_prefill_ms = prompt_tokens * 1000.0 / measured_best_tps
    required_speedup = TARGET_PREFILL_TPS / measured_best_tps
    isolated_grid_speedup = 3.28
    optimistic_amdahl = 1.0 / ((1.0 - gemm_share) + gemm_share / isolated_grid_speedup)
    projected_grid_tps = measured_best_tps * optimistic_amdahl
    macs_per_prompt = sum(float(s.get("macs") or 0.0) for s in stages)
    target_tmac_s = (macs_per_prompt / prompt_tokens) * TARGET_PREFILL_TPS / 1e12
    TARGET_ANALYSIS = {
        "best_arm": best_label, "prompt_tokens": prompt_tokens,
        "measured_tps": measured_best_tps, "target_tps": TARGET_PREFILL_TPS,
        "measured_prefill_ms": measured_prefill_ms, "target_prefill_ms": target_prefill_ms,
        "required_speedup": required_speedup, "gemm_share": gemm_share,
        "optimistic_2d_grid_tps": projected_grid_tps,
        "target_tmac_s": target_tmac_s,
        "target_fraction_of_t4_int8": target_tmac_s / T4_INT8_TMAC_S,
        "stages": stages,
    }
    print("\n15k feasibility budget:")
    print(json.dumps(TARGET_ANALYSIS, indent=2))

    pairs = [("base", "candidate")]
    if INCLUDE_R256:
        pairs.append(("base_r256", "candidate_r256"))
    for base_arm, cand_arm in pairs:
        for repeat in range(PRODUCTION_REPEATS):
            b = RESULTS / f"glbench-{repeat}-{base_arm}.json"
            c = RESULTS / f"glbench-{repeat}-{cand_arm}.json"
            p = run([BINS["candidate"]["glbench"], "compare", b, c],
                    cwd=CAND_DIR, timeout=600, check=False)
            save_log(f"compare-{repeat}-{base_arm}-vs-{cand_arm}.log", p)
            if p.returncode:
                raise SystemExit(f"glbench compare failed for {base_arm} vs {cand_arm}.")
    PROD_OK = True


## 7 · Optional Nsight Compute evidence

Kaggle images and host policies vary. If ncu is installed and hardware performance counters are permitted, the cell captures one filtered MMA launch from each diagnostic binary using supported sections. Permission denial is archived as an explicit limitation; it does not turn diagnostic timings into profiler evidence.


In [ ]:
NCU = {"available": False, "runs": {}}
ncu = shutil.which("ncu")
if not RUN_NCU:
    print("Nsight Compute disabled by configuration.")
elif ncu is None:
    print("Nsight Compute CLI is not installed in this Kaggle image.")
else:
    listed = run([ncu, "--list-sections"], timeout=300, check=False)
    save_log("ncu-list-sections.log", listed)
    available_text = listed.stdout + "\n" + listed.stderr
    wanted = [
        "LaunchStats",
        "Occupancy",
        "SpeedOfLight",
        "WarpStateStats",
        "MemoryWorkloadAnalysis",
        "ComputeWorkloadAnalysis",
    ]
    sections = [name for name in wanted if name in available_text]
    NCU["available"] = True
    NCU["sections"] = sections
    print("Nsight sections:", sections)

    for arm in ("base", "candidate"):
        report = RESULTS / f"ncu-{arm}"
        cmd = [
            ncu,
            "--target-processes", "all",
            "--kernel-name", "regex:.*gl_gemm_mma_q8.*",
            "--launch-count", "1",
            "--force-overwrite",
            "--export", report,
        ]
        for section in sections:
            cmd += ["--section", section]
        if not sections:
            cmd += ["--set", "basic"]
        cmd += [BINS[arm]["bench"]]
        p = run(cmd, cwd=BASE_DIR if arm == "base" else CAND_DIR,
                env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=14400, check=False)
        save_log(f"ncu-{arm}.log", p)
        text = p.stdout + "\n" + p.stderr
        permitted = p.returncode == 0 and "ERR_NVGPUCTRPERM" not in text
        NCU["runs"][arm] = {"returncode": p.returncode, "permitted": permitted}
        print(f"{arm:9s}: exit={p.returncode}, counters={'captured' if permitted else 'unavailable'}")
        if permitted:
            imported = run([
                ncu, "--import", str(report) + ".ncu-rep",
                "--page", "details", "--csv",
            ], timeout=1800, check=False)
            save_log(f"ncu-{arm}-details.csv", imported)


## 8 · Package the Wave 2 evidence

The final cell writes a machine-readable manifest, a concise Markdown report, and a zip archive. It does not declare an optimization win: that decision belongs at the wave gate after the production archives have been reviewed and reproduced.


In [ ]:
manifest = {
    "schema": "gwenland.glcuda.t4-ceiling.wave2.fetch.v1",
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "gpu_rows": gpu_rows,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "repo_url": REPO_URL,
    "base_rev": BASE_REV,
    "candidate_patch_sha256": PATCH_SHA256,
    "candidate_markers": markers,
    "tool_versions": tool_versions,
    "ptxas_ok": bool(globals().get("PTXAS_OK")),
    "correctness_ok": bool(globals().get("CORRECTNESS_OK")),
    "production_ok": bool(globals().get("PROD_OK")),
    "target_prefill_tps": TARGET_PREFILL_TPS,
    "force_q8_for_ptx": FORCE_Q8_FOR_PTX,
    "micro_repeats": MICRO_REPEATS,
    "production_repeats": PRODUCTION_REPEATS,
    "cold_iters": COLD_ITERS,
    "warmup_iters": WARMUP_ITERS,
    "measure_iters": MEASURE_ITERS,
    "model_path": str(model) if globals().get("PROD_OK") else None,
    "model_fetch": globals().get("MODEL_FETCH"),
    "micro_table": globals().get("MICRO_TABLE", []),
    "production_summary": globals().get("PROD_SUMMARY", []),
    "target_analysis": globals().get("TARGET_ANALYSIS"),
    "ncu": globals().get("NCU", {}),
}
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

report = [
    "# glcuda T4 Ceiling — Wave 2",
    "",
    f"- GPU: {gpu_rows[0]}",
    f"- baseline: {BASE_REV}",
    f"- candidate patch: {PATCH_SHA256}",
    f"- model: {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}",
    f"- model SHA-256: {manifest['model_fetch']['sha256'] if manifest.get('model_fetch') else 'FETCH FAILED'}",
    f"- ptxas: {'PASS' if manifest['ptxas_ok'] else 'FAIL'}",
    f"- hardware correctness: {'PASS' if manifest['correctness_ok'] else 'FAIL'}",
    f"- production glbench: {'COMPLETE' if manifest['production_ok'] else 'PENDING — model fetch or production gate failed'}",
    f"- target: {TARGET_PREFILL_TPS:.0f} prefill tok/s",
    "",
    "## Diagnostic GEMM medians",
    "",
    "| arm | shape | 8x64 us | 4x128 us | 2x256 us |",
    "|---|---|---:|---:|---:|",
]
for row in manifest["micro_table"]:
    report.append(
        f"| {row['arm']} | {row['shape']} | {row['t64']:.1f} | "
        f"{row['t128']:.1f} | {row['t256']:.1f} |"
    )

report += ["", "## Production prefill", "",
           "| arm | median of session P50 tok/s | min | max | sessions |",
           "|---|---:|---:|---:|---:|"]
for row in manifest["production_summary"]:
    report.append(
        f"| {row['arm']} | {row['session_p50_median']:.1f} | "
        f"{row['session_p50_min']:.1f} | {row['session_p50_max']:.1f} | {row['sessions']} |"
    )

if manifest["target_analysis"]:
    a = manifest["target_analysis"]
    report += [
        "", "## 15k feasibility budget", "",
        f"- best measured arm: {a['best_arm']} at {a['measured_tps']:.1f} tok/s",
        f"- required end-to-end speedup: {a['required_speedup']:.2f}x",
        f"- measured/target prompt time: {a['measured_prefill_ms']:.2f} / {a['target_prefill_ms']:.2f} ms",
        f"- measured GEMM stage share: {100*a['gemm_share']:.1f}%",
        f"- optimistic Amdahl projection using the isolated 3.28x grid result: {a['optimistic_2d_grid_tps']:.1f} tok/s",
        f"- target linear-layer demand: {a['target_tmac_s']:.2f} TMAC/s ({100*a['target_fraction_of_t4_int8']:.1f}% of 65 TMAC/s)",
    ]

report += [
    "",
    "## Interpretation rule",
    "",
    "Retain a PTX candidate only after production glbench improves by at least 5%, "
    "the result reproduces in two sessions, correctness stays green, and profiler "
    "evidence does not reveal spills or an occupancy regression that invalidates the comparison.",
    "",
    "Microbenchmark rows are diagnostic only.",
]
(RESULTS / "WAVE2_REPORT.md").write_text("\n".join(report), encoding="utf-8")

archive_base = WORK / "glcuda_t4_ceiling_wave2_fetch_results"
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RESULTS))
print(f"Results directory: {RESULTS}")
print(f"Download archive: {archive}")
print(f"Archive size: {archive.stat().st_size / 1e6:.2f} MB")
print("\n" + "\n".join(report[:30]))
